# Real-Time Multi-Modal Image Manipulation Detection

**Thesis Project** — Progressive ablation: traditional forensics (ELA) → deep multi-modal learning with ViT.

| Section | Model | Modalities |
|---------|-------|------------|
| 6 | RF (ELA) | ELA statistical features |
| 7 | Single-Stream CNN | RGB |
| 8 | Multi-Stream Gating | RGB + Noise + DCT |
| 9 | AdaptiveFusionNet | RGB + Noise + DCT |
| 10 | ViT Multi-Modal Fusion | RGB + Noise + DCT + ViT |

In [ ]:
# ============================================================
# 0. INSTALLS & IMPORTS
# ============================================================
!pip install opencv-python pillow numpy scipy scikit-learn matplotlib seaborn pywavelets transformers -q

import os, random, time, shutil
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from transformers import ViTModel

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# ---- Google Drive source paths (read-only — do not use for training I/O) ----
DRIVE_CASIA_AUTH  = "/content/drive/MyDrive/forensics_data/Unzipped images/Au"
DRIVE_CASIA_MANIP = "/content/drive/MyDrive/forensics_data/Unzipped images/Tp"
DRIVE_RAISE_AUTH  = "/content/drive/MyDrive/forensics_data/4cam/4cam_auth"
DRIVE_RAISE_MANIP = "/content/drive/MyDrive/forensics_data/4cam/4cam_splc"
DRIVE_IMD_AUTH    = "/content/drive/MyDrive/forensics_data/IMD2020/authentic"
DRIVE_IMD_MANIP   = "/content/drive/MyDrive/forensics_data/IMD2020/manipulated"
DRIVE_MODEL_DIR   = "/content/drive/MyDrive/forensics_data"

# ---- Local VM paths (Colab SSD — all training and evaluation uses these) ----
# Drive is ~3-5x slower than the local VM SSD for random image reads.
# Always copy datasets here first (Section 3A below) before building DataLoaders.
LOCAL_CASIA_AUTH  = "/content/CASIA2/Au"
LOCAL_CASIA_MANIP = "/content/CASIA2/Tp"
LOCAL_RAISE_AUTH  = "/content/RAISE/4cam_auth"
LOCAL_RAISE_MANIP = "/content/RAISE/4cam_splc"
LOCAL_IMD_AUTH    = "/content/IMD2020/authentic"
LOCAL_IMD_MANIP   = "/content/IMD2020/manipulated"

# Convenience aliases used throughout the notebook
AUTHENTIC_DIR   = LOCAL_CASIA_AUTH
MANIPULATED_DIR = LOCAL_CASIA_MANIP

print("Paths configured.")

## Section 1: Feature Extraction Utilities

ELA, Noise Residual, and DCT functions defined **once** here and reused throughout all sections.

In [ ]:
# ---- 1A. Error Level Analysis ----
def compute_ela_features(image_path, quality=95):
    """Return 12-dim ELA statistical feature dict."""
    original = Image.open(image_path)
    if original.mode != "RGB": original = original.convert("RGB")
    temp = "/tmp/ela_tmp.jpg"
    original.save(temp, "JPEG", quality=quality)
    resaved = Image.open(temp)
    ela = np.abs(np.array(original, dtype=np.float64) - np.array(resaved, dtype=np.float64))
    h, w = ela.shape[:2]
    quads = [ela[:h//2,:w//2], ela[:h//2,w//2:], ela[h//2:,:w//2], ela[h//2:,w//2:]]
    hist, _ = np.histogram(ela.flatten(), bins=5, range=(0, 255))
    hist_norm = hist / (hist.sum() + 1e-10)
    edges = cv2.Canny(np.array(original, dtype=np.uint8), 100, 200)
    mask = edges > 0
    feat = {
        "mean_error": np.mean(ela), "std_error": np.std(ela),
        "max_error": np.max(ela),   "min_error": np.min(ela),
        "q25_error": np.percentile(ela, 25), "q50_error": np.percentile(ela, 50),
        "q75_error": np.percentile(ela, 75),
        "spatial_variance": np.std([np.mean(q) for q in quads]),
        "edge_error_ratio": ela[mask].mean() / (ela[~mask].mean() + 1e-10) if mask.sum() > 0 else 1.0,
    }
    for i, h_ in enumerate(hist_norm): feat[f"hist_bin_{i}"] = h_
    if os.path.exists(temp): os.remove(temp)
    return feat

# ---- 1B. Noise Residual ----
def get_noise_residual(img):
    """High-frequency noise via Gaussian blur subtraction."""
    arr = np.array(img).astype(np.float32)
    return Image.fromarray(np.clip(arr - cv2.GaussianBlur(arr, (5, 5), 0), 0, 255).astype(np.uint8))

# ---- 1C. DCT Log-Magnitude ----
def get_dct_image(img):
    """Log-magnitude DCT of grayscale; returned as 3-channel PIL image."""
    gray = np.float32(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)) / 255.0
    dct  = cv2.normalize(np.log(np.abs(cv2.dct(gray)) + 1e-6), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return Image.fromarray(cv2.cvtColor(dct, cv2.COLOR_GRAY2RGB))

# ---- 1D. ELA as Visual Modality (for CNN input) ----
def get_ela_modality(img, quality=95):
    """
    Compute ELA from a PIL Image and return as a 3-channel PIL image
    suitable for CNN input (same interface as get_noise_residual / get_dct_image).
    """
    tmp = "/tmp/ela_mod.jpg"
    img.save(tmp, "JPEG", quality=quality)
    resaved  = Image.open(tmp)
    ela      = np.abs(np.array(img, dtype=np.float64) - np.array(resaved, dtype=np.float64))
    ela_gray = np.mean(ela, axis=2)
    ela_norm = ((ela_gray - ela_gray.min()) /
                (ela_gray.max() - ela_gray.min() + 1e-10) * 255).astype(np.uint8)
    if os.path.exists(tmp): os.remove(tmp)
    return Image.fromarray(cv2.cvtColor(ela_norm, cv2.COLOR_GRAY2RGB))

## Section 2: Dataset Classes

In [ ]:
VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")


class ForensicsDataset(Dataset):
    """RGB-only dataset for single-stream CNN."""
    def __init__(self, auth_dir, manip_dir, transform=None):
        self.transform = transform
        self.samples = (
            [(os.path.join(auth_dir,  f), 0) for f in os.listdir(auth_dir)  if f.lower().endswith(VALID_EXT)] +
            [(os.path.join(manip_dir, f), 1) for f in os.listdir(manip_dir) if f.lower().endswith(VALID_EXT)]
        )
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return (self.transform(img) if self.transform else img), label


class MultiModalDataset(Dataset):
    """Returns (rgb, noise_residual, dct_log, label) per image."""
    def __init__(self, auth_dir, manip_dir, transform):
        self.transform = transform
        self.samples = (
            [(os.path.join(auth_dir,  f), 0) for f in os.listdir(auth_dir)  if f.lower().endswith(VALID_EXT)] +
            [(os.path.join(manip_dir, f), 1) for f in os.listdir(manip_dir) if f.lower().endswith(VALID_EXT)]
        )
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self.samples))
        return (
            self.transform(img),
            self.transform(get_noise_residual(img)),
            self.transform(get_dct_image(img)),
            label
        )

class MultiModal5Dataset(Dataset):
    """
    5-stream dataset: returns (rgb, ela, noise, dct, label).
    Used by CrossModalFusionNet (Option A+B — novel contribution).
    """
    def __init__(self, auth_dir, manip_dir, transform):
        self.transform = transform
        self.samples = (
            [(os.path.join(auth_dir,  f), 0) for f in os.listdir(auth_dir)  if f.lower().endswith(VALID_EXT)] +
            [(os.path.join(manip_dir, f), 1) for f in os.listdir(manip_dir) if f.lower().endswith(VALID_EXT)]
        )
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self.samples))
        return (
            self.transform(img),
            self.transform(get_ela_modality(img)),
            self.transform(get_noise_residual(img)),
            self.transform(get_dct_image(img)),
            label
        )

## Section 3: Dataset Loading & Splits

- **CASIA2**: primary training dataset (70/15/15 split)
- **RAISE**, **IMD2020**: cross-dataset generalization (eval only, no fine-tuning)

In [ ]:
# ============================================================
# 3A. Copy datasets from Drive to local VM SSD (run once per session)
# ============================================================
# Drive FUSE reads are slow (~50-100 MB/s). VM SSD is 3-5x faster
# for the many small random image reads that DataLoader workers do.
# shutil.copytree skips a directory if it already exists and is non-empty,
# so this is safe to re-run and only copies what is missing.

def copy_to_local(src, dst, label):
    """Copy src directory to dst if dst is empty or missing."""
    if os.path.isdir(dst) and len(os.listdir(dst)) > 0:
        print(f"  {label}: already local ({len(os.listdir(dst))} files) — skipping")
        return
    os.makedirs(dst, exist_ok=True)
    print(f"  {label}: copying {src} → {dst} ...")
    # copy contents (not the folder itself) so dst structure stays clean
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print(f"  {label}: done ({len(os.listdir(dst))} files)")

print("Copying datasets to local VM...")
copy_to_local(DRIVE_CASIA_AUTH,  LOCAL_CASIA_AUTH,  "CASIA2 Authentic")
copy_to_local(DRIVE_CASIA_MANIP, LOCAL_CASIA_MANIP, "CASIA2 Tampered")
copy_to_local(DRIVE_RAISE_AUTH,  LOCAL_RAISE_AUTH,  "RAISE Authentic")
copy_to_local(DRIVE_RAISE_MANIP, LOCAL_RAISE_MANIP, "RAISE Spliced")
copy_to_local(DRIVE_IMD_AUTH,    LOCAL_IMD_AUTH,    "IMD2020 Authentic")
copy_to_local(DRIVE_IMD_MANIP,   LOCAL_IMD_MANIP,   "IMD2020 Manipulated")
print("All datasets ready.")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ---- CASIA2: primary training dataset (70 / 15 / 15 split) ----
full_dataset = MultiModalDataset(LOCAL_CASIA_AUTH, LOCAL_CASIA_MANIP, train_transform)
train_sz = int(0.70 * len(full_dataset))
val_sz   = int(0.15 * len(full_dataset))
test_sz  = len(full_dataset) - train_sz - val_sz
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_sz, val_sz, test_sz],
    generator=torch.Generator().manual_seed(SEED)
)

BATCH = 16
# num_workers=0: DataLoader runs in the main process so notebook-defined
# functions (get_noise_residual, get_dct_image, get_ela_modality) stay in scope.
# With data on local VM SSD this has negligible performance impact.
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

# ---- Cross-dataset loaders (eval only, no fine-tuning) ----
RAISE_dataset = MultiModalDataset(LOCAL_RAISE_AUTH, LOCAL_RAISE_MANIP, val_transform)
IMD_dataset   = MultiModalDataset(LOCAL_IMD_AUTH,   LOCAL_IMD_MANIP,   val_transform)
RAISE_loader  = DataLoader(RAISE_dataset, batch_size=BATCH, shuffle=False)
IMD_loader    = DataLoader(IMD_dataset,   batch_size=BATCH, shuffle=False)

print(f"CASIA2  — Train:{len(train_dataset)} | Val:{len(val_dataset)} | Test:{len(test_dataset)}")
print(f"RAISE   — {len(RAISE_dataset)} images")
print(f"IMD2020 — {len(IMD_dataset)} images")
_lbl = np.array([full_dataset.samples[i][1] for i in train_dataset.indices])
print(f"Class balance (train) — Auth:{(_lbl==0).sum()} | Manip:{(_lbl==1).sum()}")

# ---- 5-stream datasets (CrossModalFusionNet — Options A+B) ----
full5_dataset = MultiModal5Dataset(LOCAL_CASIA_AUTH, LOCAL_CASIA_MANIP, train_transform)
train5_sz = int(0.70 * len(full5_dataset))
val5_sz   = int(0.15 * len(full5_dataset))
test5_sz  = len(full5_dataset) - train5_sz - val5_sz
train5_dataset, val5_dataset, test5_dataset = random_split(
    full5_dataset, [train5_sz, val5_sz, test5_sz],
    generator=torch.Generator().manual_seed(SEED)
)
train5_loader = DataLoader(train5_dataset, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val5_loader   = DataLoader(val5_dataset,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
test5_loader  = DataLoader(test5_dataset,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

RAISE5_dataset = MultiModal5Dataset(LOCAL_RAISE_AUTH, LOCAL_RAISE_MANIP, val_transform)
IMD5_dataset   = MultiModal5Dataset(LOCAL_IMD_AUTH,   LOCAL_IMD_MANIP,   val_transform)
RAISE5_loader  = DataLoader(RAISE5_dataset, batch_size=BATCH, shuffle=False)
IMD5_loader    = DataLoader(IMD5_dataset,   batch_size=BATCH, shuffle=False)
print(f"5-stream — Train:{len(train5_dataset)} | RAISE:{len(RAISE5_dataset)} | IMD:{len(IMD5_dataset)}")

## Section 4: Visualization

In [ ]:
def visualize_modalities(dataset, idx=0, title="Sample"):
    rgb, noise, dct, label = dataset[idx]
    def to_np(t):
        t = t.permute(1,2,0).numpy()
        return (t - t.min()) / (t.max() - t.min() + 1e-6)
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(to_np(rgb));   ax[0].set_title("RGB");            ax[0].axis("off")
    ax[1].imshow(to_np(noise)); ax[1].set_title("Noise Residual"); ax[1].axis("off")
    ax[2].imshow(to_np(dct));   ax[2].set_title("DCT Log-Mag");   ax[2].axis("off")
    lbl = "Manipulated" if label == 1 else "Authentic"
    fig.suptitle(f"{title}  |  {lbl}", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

if len(full_dataset)  > 0: visualize_modalities(full_dataset,  title="CASIA2")
if len(RAISE_dataset) > 0: visualize_modalities(RAISE_dataset, title="RAISE")

## Section 5: Shared Evaluation Function

One canonical `evaluate_model()` used by **all** models. Uses optimal ROC threshold instead of hardcoded 0.5.

In [ ]:
def evaluate_model(model, loader, name="Model", single_stream=False, five_stream=False):
    """
    Universal evaluator.
    single_stream=True  → (img, label) batches
    five_stream=True    → (rgb, ela, noise, dct, label) batches
    default             → (rgb, noise, dct, label) batches
    Returns: acc, precision, recall, f1, auc, cm
    """
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Eval {name}"):
            if single_stream:
                imgs, labels = batch
                logits = model(imgs.to(device))
            elif five_stream:
                rgb, ela, noise, dct, labels = batch
                logits = model(rgb.to(device), ela.to(device), noise.to(device), dct.to(device))
            else:
                rgb, noise, dct, labels = batch
                logits = model(rgb.to(device), noise.to(device), dct.to(device))
            all_probs.extend(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_labels.extend(labels.numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    opt_thr = thresholds[np.argmax(tpr - fpr)]
    auc     = roc_auc_score(all_labels, all_probs)
    preds   = (all_probs >= opt_thr).astype(int)
    acc   = accuracy_score(all_labels, preds)
    prec  = precision_score(all_labels, preds, zero_division=0)
    rec   = recall_score(all_labels, preds, zero_division=0)
    f1    = f1_score(all_labels, preds, zero_division=0)
    cm    = confusion_matrix(all_labels, preds)
    print(f"\n===== {name} =====")
    print(f"  Threshold (optimal): {opt_thr:.4f}")
    print(f"  Accuracy : {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")
    print(classification_report(all_labels, preds, target_names=["Authentic","Manipulated"], digits=4))
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Authentic","Manipulated"], yticklabels=["Authentic","Manipulated"])
    plt.title(f"{name} — Confusion Matrix"); plt.tight_layout(); plt.show()
    return acc, prec, rec, f1, auc, cm

## Section 6: Traditional Baseline — ELA + Random Forest

In [ ]:
def extract_ela_batch(image_dir, label):
    records = []
    for fname in tqdm(os.listdir(image_dir), desc=f"ELA {image_dir[-20:]}"):
        if not fname.lower().endswith(VALID_EXT): continue
        try:
            feat = compute_ela_features(os.path.join(image_dir, fname))
            feat["label"] = label
            records.append(feat)
        except Exception as e:
            print(f"  Skip {fname}: {e}")
    return pd.DataFrame(records)

auth_df  = extract_ela_batch(AUTHENTIC_DIR,   0)
manip_df = extract_ela_batch(MANIPULATED_DIR, 1)
ela_df = pd.concat([auth_df, manip_df], ignore_index=True).dropna()

feat_cols = [col for col in ela_df.columns if col != "label"]
X, y = ela_df[feat_cols].values, ela_df["label"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)
y_prob = rf.predict_proba(X_te)[:, 1]

rf_acc  = accuracy_score(y_te, y_pred)
rf_prec = precision_score(y_te, y_pred, zero_division=0)
rf_rec  = recall_score(y_te, y_pred, zero_division=0)
rf_f1   = f1_score(y_te, y_pred, zero_division=0)
rf_auc  = roc_auc_score(y_te, y_prob)

print(f"\n===== RF (ELA) Baseline =====")
print(f"  Acc:{rf_acc:.4f}  Prec:{rf_prec:.4f}  Rec:{rf_rec:.4f}  F1:{rf_f1:.4f}  AUC:{rf_auc:.4f}")
print(classification_report(y_te, y_pred, target_names=["Authentic","Manipulated"]))
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_te, y_pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=["Authentic","Manipulated"], yticklabels=["Authentic","Manipulated"])
plt.title("RF (ELA) Confusion Matrix"); plt.tight_layout(); plt.show()

## Section 7: Single-Stream CNN (EfficientNet-B0, RGB only)

Baseline deep learning model — no forensic streams.

In [ ]:
ss_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
ss_dataset = ForensicsDataset(AUTHENTIC_DIR, MANIPULATED_DIR, ss_transform)
ss_tr_sz = int(0.70*len(ss_dataset)); ss_v_sz = int(0.15*len(ss_dataset))
ss_tr, ss_v, ss_te = random_split(
    ss_dataset, [ss_tr_sz, ss_v_sz, len(ss_dataset)-ss_tr_sz-ss_v_sz],
    generator=torch.Generator().manual_seed(SEED)
)
ss_train_loader = DataLoader(ss_tr, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
ss_val_loader   = DataLoader(ss_v,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)
ss_test_loader  = DataLoader(ss_te, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

ss_lbls = np.array([ss_dataset.samples[i][1] for i in ss_tr.indices])
ss_wts  = torch.tensor(len(ss_lbls)/(2.0*np.bincount(ss_lbls)), dtype=torch.float32).to(device)

ss_model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
ss_model.classifier[1] = nn.Linear(ss_model.classifier[1].in_features, 2)
ss_model = ss_model.to(device)

ss_crit = nn.CrossEntropyLoss(weight=ss_wts, label_smoothing=0.05)
ss_opt  = optim.Adam(ss_model.parameters(), lr=3e-4, weight_decay=1e-4)
ss_sch  = optim.lr_scheduler.CosineAnnealingLR(ss_opt, T_max=20)

best_ss_f1, ss_pat = 0, 0
for epoch in range(20):
    ss_model.train()
    for imgs, lbls in tqdm(ss_train_loader, desc=f"SS Epoch {epoch+1}"):
        ss_opt.zero_grad()
        ss_crit(ss_model(imgs.to(device)), lbls.to(device)).backward()
        ss_opt.step()
    ss_model.eval(); vp, vl = [], []
    with torch.no_grad():
        for imgs, lbls in ss_val_loader:
            vp.extend(torch.argmax(ss_model(imgs.to(device)), 1).cpu().numpy()); vl.extend(lbls.numpy())
    vf1 = f1_score(vl, vp); ss_sch.step()
    print(f"Epoch {epoch+1} | Val F1: {vf1:.4f}")
    if vf1 > best_ss_f1:
        best_ss_f1 = vf1; torch.save(ss_model.state_dict(), f"{DRIVE_MODEL_DIR}/best_single_stream.pth"); ss_pat = 0
    else:
        ss_pat += 1
    if ss_pat >= 3: print("Early stop."); break
print(f"Best SS F1: {best_ss_f1:.4f}")

In [ ]:
ss_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_single_stream.pth", map_location=device))
ss_acc, ss_prec, ss_rec, ss_f1, ss_auc, ss_cm = evaluate_model(
    ss_model, ss_test_loader, "Single-Stream CNN", single_stream=True)

## Section 8: Multi-Stream CNN with Channel Gating

Three EfficientNet-B0 streams (RGB, Noise Residual, DCT Log-Mag) with per-channel sigmoid gates before concatenation fusion.

In [ ]:
class MultiModalFusionNet(nn.Module):
    """3-stream CNN with sigmoid channel gating before concat fusion."""
    def __init__(self, freeze_until=1):
        super().__init__()
        W = EfficientNet_B0_Weights.DEFAULT
        self.rgb_bb, self.noise_bb, self.dct_bb = (
            efficientnet_b0(weights=W), efficientnet_b0(weights=W), efficientnet_b0(weights=W))
        for b in (self.rgb_bb, self.noise_bb, self.dct_bb):
            b.classifier = nn.Identity()
            for i, ch in enumerate(b.features.children()):
                if i < freeze_until:
                    for p in ch.parameters(): p.requires_grad = False
        self.rgb_gate   = nn.Sequential(nn.Linear(1280, 1280), nn.Sigmoid())
        self.noise_gate = nn.Sequential(nn.Linear(1280, 1280), nn.Sigmoid())
        self.dct_gate   = nn.Sequential(nn.Linear(1280, 1280), nn.Sigmoid())
        self.fc = nn.Sequential(
            nn.Linear(1280 * 3, 512), nn.ReLU(), nn.Dropout(0.4), nn.Linear(512, 2))
    def forward(self, rgb, noise, dct):
        f1 = self.rgb_bb(rgb);     f1 = f1 * self.rgb_gate(f1)
        f2 = self.noise_bb(noise); f2 = f2 * self.noise_gate(f2)
        f3 = self.dct_bb(dct);     f3 = f3 * self.dct_gate(f3)
        return self.fc(torch.cat([f1, f2, f3], dim=1))

In [ ]:
gating_model = MultiModalFusionNet(freeze_until=1).to(device)
g_crit = nn.CrossEntropyLoss(label_smoothing=0.05)
g_opt  = optim.Adam(gating_model.parameters(), lr=2e-4, weight_decay=1e-4)
g_sch  = optim.lr_scheduler.ReduceLROnPlateau(g_opt, mode="max", factor=0.5, patience=2)

best_g_f1, g_pat = 0, 0
for epoch in range(20):
    gating_model.train()
    for rgb, noise, dct, lbls in tqdm(train_loader, desc=f"Gating E{epoch+1}"):
        g_opt.zero_grad()
        g_crit(gating_model(rgb.to(device), noise.to(device), dct.to(device)), lbls.to(device)).backward()
        g_opt.step()
    gating_model.eval(); vp, vl = [], []
    with torch.no_grad():
        for rgb, noise, dct, lbls in val_loader:
            vp.extend(torch.argmax(gating_model(rgb.to(device), noise.to(device), dct.to(device)), 1).cpu().numpy())
            vl.extend(lbls.numpy())
    vf1 = f1_score(vl, vp); g_sch.step(vf1)
    print(f"Epoch {epoch+1} | Val F1: {vf1:.4f}")
    if vf1 > best_g_f1:
        best_g_f1 = vf1; torch.save(gating_model.state_dict(), f"{DRIVE_MODEL_DIR}/best_gating_fusion.pth"); g_pat = 0
    else:
        g_pat += 1
    if g_pat >= 3: print("Early stop."); break
print(f"Best Gating F1: {best_g_f1:.4f}")

In [ ]:
gating_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_gating_fusion.pth", map_location=device))
g_acc, g_prec, g_rec, g_f1, g_auc, g_cm = evaluate_model(gating_model, test_loader, "3-Stream Gating")

## Section 9: AdaptiveFusionNet — Learnable Attention Fusion

Replaces fixed gating with sample-adaptive softmax attention weights. The model learns *which* forensic modality is most informative per image.

In [ ]:
class AdaptiveFusionNet(nn.Module):
    """3-stream CNN with learnable softmax attention over modalities."""
    def __init__(self, freeze_until=4):
        super().__init__()
        W = EfficientNet_B0_Weights.DEFAULT
        self.rgb_bb, self.noise_bb, self.dct_bb = (
            efficientnet_b0(weights=W), efficientnet_b0(weights=W), efficientnet_b0(weights=W))
        for b in (self.rgb_bb, self.noise_bb, self.dct_bb):
            b.classifier = nn.Identity()
            for i, ch in enumerate(b.features.children()):
                if i < freeze_until:
                    for p in ch.parameters(): p.requires_grad = False
        self.attention  = nn.Sequential(nn.Linear(1280 * 3, 256), nn.ReLU(), nn.Linear(256, 3))
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, 2))
    def forward(self, rgb, noise, dct):
        fr = self.rgb_bb(rgb); fn = self.noise_bb(noise); fd = self.dct_bb(dct)
        w  = torch.softmax(self.attention(torch.cat([fr, fn, fd], dim=1)), dim=1)
        return self.classifier(w[:, 0:1]*fr + w[:, 1:2]*fn + w[:, 2:3]*fd)

In [ ]:
adaptive_model = AdaptiveFusionNet(freeze_until=4).to(device)
a_crit = nn.CrossEntropyLoss(label_smoothing=0.05)
a_opt  = optim.Adam(adaptive_model.parameters(), lr=2e-4, weight_decay=1e-4)
a_sch  = optim.lr_scheduler.ReduceLROnPlateau(a_opt, mode="max", factor=0.5, patience=2)

best_a_f1, a_pat = 0, 0
for epoch in range(20):
    adaptive_model.train()
    for rgb, noise, dct, lbls in tqdm(train_loader, desc=f"Adaptive E{epoch+1}"):
        a_opt.zero_grad()
        a_crit(adaptive_model(rgb.to(device), noise.to(device), dct.to(device)), lbls.to(device)).backward()
        a_opt.step()
    adaptive_model.eval(); vp, vl = [], []
    with torch.no_grad():
        for rgb, noise, dct, lbls in val_loader:
            vp.extend(torch.argmax(adaptive_model(rgb.to(device), noise.to(device), dct.to(device)), 1).cpu().numpy())
            vl.extend(lbls.numpy())
    vf1 = f1_score(vl, vp); a_sch.step(vf1)
    print(f"Epoch {epoch+1} | Val F1: {vf1:.4f}")
    if vf1 > best_a_f1:
        best_a_f1 = vf1; torch.save(adaptive_model.state_dict(), f"{DRIVE_MODEL_DIR}/best_adaptive_fusion.pth"); a_pat = 0
    else:
        a_pat += 1
    if a_pat >= 3: print("Early stop."); break
print(f"Best Adaptive F1: {best_a_f1:.4f}")

In [ ]:
adaptive_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_adaptive_fusion.pth", map_location=device))
a_acc, a_prec, a_rec, a_f1, a_auc, a_cm = evaluate_model(adaptive_model, test_loader, "AdaptiveFusionNet")

## Section 10: ViT + Multi-Stream Attention Fusion (Main Contribution)

**Architecture:** 4 parallel streams fused via sample-adaptive softmax attention.

| Stream | Backbone | Input | Output dim |
|--------|---------|-------|------------|
| 1 | EfficientNet-B0 | RGB | 1280 |
| 2 | EfficientNet-B0 | Noise Residual | 1280 |
| 3 | EfficientNet-B0 | DCT Log-Mag | 1280 |
| 4 | ViT-B/16 | RGB | 768 → 1280 (projected) |

Differential learning rates: CNN streams at 2e-5, ViT at 5e-6 (more conservative to preserve pretrained representations).

In [ ]:
class ViTFeatureExtractor(nn.Module):
    """ViT-B/16 — returns CLS token (B, 768). Last N encoder blocks unfrozen."""
    def __init__(self, unfreeze_last_blocks=2):
        super().__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224")
        for p in self.vit.parameters(): p.requires_grad = False
        for p in self.vit.encoder.layer[-unfreeze_last_blocks:].parameters(): p.requires_grad = True
    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]  # CLS token


class ViTMultiModalFusionNet(nn.Module):
    """
    4-stream multi-modal fusion.
    Streams: RGB + Noise + DCT (EfficientNet-B0) + ViT (RGB).
    Fusion: sample-adaptive softmax attention over all 4 modalities.
    """
    def __init__(self, freeze_until=1):
        super().__init__()
        W = EfficientNet_B0_Weights.DEFAULT
        self.rgb_bb, self.noise_bb, self.dct_bb = (
            efficientnet_b0(weights=W), efficientnet_b0(weights=W), efficientnet_b0(weights=W))
        for b in (self.rgb_bb, self.noise_bb, self.dct_bb):
            b.classifier = nn.Identity()
            for i, ch in enumerate(b.features.children()):
                if i < freeze_until:
                    for p in ch.parameters(): p.requires_grad = False
        self.vit      = ViTFeatureExtractor(unfreeze_last_blocks=2)
        self.vit_proj = nn.Linear(768, 1280)
        self.attention  = nn.Sequential(nn.Linear(1280 * 4, 256), nn.ReLU(), nn.Linear(256, 4))
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, 2))
    def forward(self, rgb, noise, dct):
        fr = self.rgb_bb(rgb)
        fn = self.noise_bb(noise)
        fd = self.dct_bb(dct)
        fv = self.vit_proj(self.vit(rgb))
        w  = torch.softmax(self.attention(torch.cat([fr, fn, fd, fv], dim=1)), dim=1)
        fused = w[:, 0:1]*fr + w[:, 1:2]*fn + w[:, 2:3]*fd + w[:, 3:4]*fv
        return self.classifier(fused)

In [ ]:
vit_model = ViTMultiModalFusionNet(freeze_until=1).to(device)

# Class-balanced loss
_lbl  = np.array([full_dataset.samples[i][1] for i in train_dataset.indices])
_wts  = torch.tensor(len(_lbl)/(2.0*np.bincount(_lbl)), dtype=torch.float32).to(device)
v_crit = nn.CrossEntropyLoss(weight=_wts, label_smoothing=0.05)

# Differential learning rates: ViT gets lower LR to preserve pretrained representations
v_opt = torch.optim.AdamW([
    {"params": vit_model.rgb_bb.parameters(),    "lr": 2e-5},
    {"params": vit_model.noise_bb.parameters(),  "lr": 2e-5},
    {"params": vit_model.dct_bb.parameters(),    "lr": 2e-5},
    {"params": vit_model.vit.parameters(),       "lr": 5e-6},
    {"params": vit_model.vit_proj.parameters(),  "lr": 5e-5},
    {"params": vit_model.attention.parameters(), "lr": 5e-5},
    {"params": vit_model.classifier.parameters(),"lr": 5e-5},
], weight_decay=1e-4)
v_sch = optim.lr_scheduler.ReduceLROnPlateau(v_opt, mode="max", factor=0.3, patience=3)

best_v_f1, v_pat = 0, 0
for epoch in range(20):
    vit_model.train(); ep_loss = 0
    for rgb, noise, dct, lbls in tqdm(train_loader, desc=f"ViT E{epoch+1}"):
        v_opt.zero_grad()
        loss = v_crit(vit_model(rgb.to(device), noise.to(device), dct.to(device)), lbls.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vit_model.parameters(), 1.0)  # gradient clipping
        v_opt.step(); ep_loss += loss.item()
    vit_model.eval(); vp, vl = [], []
    with torch.no_grad():
        for rgb, noise, dct, lbls in val_loader:
            vp.extend(torch.argmax(vit_model(rgb.to(device), noise.to(device), dct.to(device)), 1).cpu().numpy())
            vl.extend(lbls.numpy())
    vf1 = f1_score(vl, vp); v_sch.step(vf1)
    print(f"Epoch {epoch+1} | Loss:{ep_loss/len(train_loader):.4f} | Val F1:{vf1:.4f}")
    if vf1 > best_v_f1:
        best_v_f1 = vf1; torch.save(vit_model.state_dict(), f"{DRIVE_MODEL_DIR}/best_vit_fusion.pth"); v_pat = 0
    else:
        v_pat += 1
    if v_pat >= 6: print("Early stop."); break
torch.save(vit_model.state_dict(), f"{DRIVE_MODEL_DIR}/last_vit_fusion.pth")
print(f"Best ViT Fusion F1: {best_v_f1:.4f}")

In [ ]:
vit_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_vit_fusion.pth", map_location=device))
v_acc, v_prec, v_rec, v_f1, v_auc, v_cm = evaluate_model(vit_model, test_loader, "ViT Fusion (4-stream)")

## Section 10b: Novel Contribution — Cross-Modal Transformer Fusion with ELA Stream

**Options A + B combined.**

### What makes this novel:
- **Option B**: ELA is introduced as a *learned visual stream* (5th CNN backbone), not just hand-crafted RF features. The model learns which ELA patterns indicate manipulation end-to-end.
- **Option A**: Instead of scalar softmax weights over concatenated features, a **transformer encoder over the modality dimension** is used. Each stream attends to all other streams — RGB can suppress noise features when they are uninformative, ELA can amplify DCT signals that co-occur with compression anomalies, etc.

### Architecture
```
Stream 1: RGB   → EfficientNet-B0 → (B, 1280)
Stream 2: ELA   → EfficientNet-B0 → (B, 1280)   ← NEW (Option B)
Stream 3: Noise → EfficientNet-B0 → (B, 1280)
Stream 4: DCT   → EfficientNet-B0 → (B, 1280)
Stream 5: ViT   → ViT-B/16 + proj → (B, 1280)
                                         ↓
              Stack: (B, 5, 1280)
                                         ↓
    Modality Transformer Encoder (2 layers, 8 heads)  ← NEW (Option A)
    Each modality attends to all others
                                         ↓
              Mean pool: (B, 1280)
                                         ↓
              Classifier: 1280 → 256 → 2
```

**Claim:** "We propose cross-modal transformer fusion that models inter-modality dependencies — each forensic stream (ELA, noise, DCT) contextualizes the others — outperforming both scalar attention and fixed concatenation baselines."

In [ ]:
class ModalityTransformerEncoder(nn.Module):
    """
    Transformer encoder applied over the modality dimension.
    Input:  (B, N, D) — B=batch, N=num modalities, D=feature dim
    Output: (B, D)    — mean-pooled attended features
    Each modality attends to all others via multi-head self-attention.
    """
    def __init__(self, d_model=1280, n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        # Learnable modality position embeddings so transformer knows which stream is which
        self.modality_pos = nn.Parameter(torch.randn(1, 5, d_model) * 0.02)

    def forward(self, x):  # x: (B, N, D)
        x = x + self.modality_pos  # add positional embedding per modality
        out = self.transformer(x)  # (B, N, D)
        return out.mean(dim=1)     # (B, D) — mean pool over modalities


class CrossModalFusionNet(nn.Module):
    """
    Novel 5-stream + ViT architecture with cross-modal transformer fusion.

    Streams: RGB, ELA, Noise, DCT (EfficientNet-B0) + ViT (RGB).
    Fusion:  Transformer encoder over modality dimension — cross-modal attention.
    """
    def __init__(self, freeze_until=1, d_model=1280):
        super().__init__()
        W = EfficientNet_B0_Weights.DEFAULT
        self.rgb_bb   = efficientnet_b0(weights=W)
        self.ela_bb   = efficientnet_b0(weights=W)
        self.noise_bb = efficientnet_b0(weights=W)
        self.dct_bb   = efficientnet_b0(weights=W)
        for b in (self.rgb_bb, self.ela_bb, self.noise_bb, self.dct_bb):
            b.classifier = nn.Identity()
            for i, ch in enumerate(b.features.children()):
                if i < freeze_until:
                    for p in ch.parameters(): p.requires_grad = False

        self.vit      = ViTFeatureExtractor(unfreeze_last_blocks=2)
        self.vit_proj = nn.Linear(768, d_model)

        self.modality_transformer = ModalityTransformerEncoder(
            d_model=d_model, n_heads=8, n_layers=2, dropout=0.1
        )
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, 2)
        )

    def forward(self, rgb, ela, noise, dct):
        f_rgb   = self.rgb_bb(rgb)
        f_ela   = self.ela_bb(ela)
        f_noise = self.noise_bb(noise)
        f_dct   = self.dct_bb(dct)
        f_vit   = self.vit_proj(self.vit(rgb))

        # Stack modalities as a sequence: (B, 5, D)
        feats = torch.stack([f_rgb, f_ela, f_noise, f_dct, f_vit], dim=1)

        # Cross-modal transformer: each modality attends to all others
        fused = self.modality_transformer(feats)  # (B, D)

        return self.classifier(fused)

In [ ]:
# Quick sanity-check forward pass
cross_model = CrossModalFusionNet(freeze_until=1).to(device)
rgb_s, ela_s, noise_s, dct_s, _ = train5_dataset[0]
with torch.no_grad():
    out = cross_model(
        rgb_s.unsqueeze(0).to(device), ela_s.unsqueeze(0).to(device),
        noise_s.unsqueeze(0).to(device), dct_s.unsqueeze(0).to(device)
    )
print("Output shape:", out.shape)  # expected: (1, 2)

In [ ]:
# Class-balanced loss
_lbl5  = np.array([full5_dataset.samples[i][1] for i in train5_dataset.indices])
_wts5  = torch.tensor(len(_lbl5)/(2.0*np.bincount(_lbl5)), dtype=torch.float32).to(device)
x_crit = nn.CrossEntropyLoss(weight=_wts5, label_smoothing=0.05)

# Differential learning rates: ViT much lower, transformer fusion layers moderate
x_opt = torch.optim.AdamW([
    {"params": cross_model.rgb_bb.parameters(),              "lr": 2e-5},
    {"params": cross_model.ela_bb.parameters(),              "lr": 2e-5},
    {"params": cross_model.noise_bb.parameters(),            "lr": 2e-5},
    {"params": cross_model.dct_bb.parameters(),              "lr": 2e-5},
    {"params": cross_model.vit.parameters(),                 "lr": 5e-6},
    {"params": cross_model.vit_proj.parameters(),            "lr": 5e-5},
    {"params": cross_model.modality_transformer.parameters(),"lr": 5e-5},
    {"params": cross_model.classifier.parameters(),          "lr": 5e-5},
], weight_decay=1e-4)
x_sch = optim.lr_scheduler.ReduceLROnPlateau(x_opt, mode="max", factor=0.3, patience=3)

best_x_f1, x_pat = 0, 0
for epoch in range(20):
    cross_model.train(); ep_loss = 0
    for rgb, ela, noise, dct, lbls in tqdm(train5_loader, desc=f"CrossModal E{epoch+1}"):
        x_opt.zero_grad()
        loss = x_crit(
            cross_model(rgb.to(device), ela.to(device), noise.to(device), dct.to(device)),
            lbls.to(device)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cross_model.parameters(), 1.0)
        x_opt.step(); ep_loss += loss.item()
    cross_model.eval(); vp, vl = [], []
    with torch.no_grad():
        for rgb, ela, noise, dct, lbls in val5_loader:
            vp.extend(torch.argmax(
                cross_model(rgb.to(device), ela.to(device), noise.to(device), dct.to(device)), 1
            ).cpu().numpy())
            vl.extend(lbls.numpy())
    vf1 = f1_score(vl, vp); x_sch.step(vf1)
    print(f"Epoch {epoch+1} | Loss:{ep_loss/len(train5_loader):.4f} | Val F1:{vf1:.4f}")
    if vf1 > best_x_f1:
        best_x_f1 = vf1
        torch.save(cross_model.state_dict(), f"{DRIVE_MODEL_DIR}/best_cross_modal.pth")
        x_pat = 0
    else:
        x_pat += 1
    if x_pat >= 6: print("Early stop."); break
torch.save(cross_model.state_dict(), f"{DRIVE_MODEL_DIR}/last_cross_modal.pth")
print(f"Best CrossModal F1: {best_x_f1:.4f}")

In [ ]:
cross_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_cross_modal.pth", map_location=device))
x_acc, x_prec, x_rec, x_f1, x_auc, x_cm = evaluate_model(
    cross_model, test5_loader, "CrossModal Fusion (A+B)", five_stream=True)

## Section 11: Cross-Dataset Generalization

Evaluate all deep multi-modal models on **RAISE** and **IMD2020** without any fine-tuning. Tests whether learned forensic representations generalize beyond CASIA2.

---

### Resuming in a new session?

Run the following cells first (they define classes and functions but do no training):

| Cell | Content |
|------|---------|
| Section 0 | Imports, path constants, device |
| Section 1 | `compute_ela_features`, `get_noise_residual`, `get_dct_image`, `get_ela_modality` |
| Section 2 | `ForensicsDataset`, `MultiModalDataset`, `MultiModal5Dataset` |
| Section 5 | `evaluate_model()` |
| Section 8 | `MultiModalFusionNet` |
| Section 9 | `AdaptiveFusionNet` |
| Section 10 | `ViTFeatureExtractor`, `ViTMultiModalFusionNet` |
| Section 10b | `ModalityTransformerEncoder`, `CrossModalFusionNet` |

Then run Section 11 directly — it will load all model weights from Drive and rebuild the evaluation loaders.

In [ ]:
# ============================================================
# Section 11: Cross-Dataset Generalization (session-safe)
# ============================================================
# Self-contained for a fresh session: rebuilds loaders and loads
# all model weights from Drive. No training cells need to be run.
# Prerequisites: Section 0, 1, 2, 5, 8, 9, 10, 10b (definition cells only).
# ============================================================

# ---- Rebuild cross-dataset loaders ----
_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

RAISE_dataset  = MultiModalDataset(LOCAL_RAISE_AUTH,  LOCAL_RAISE_MANIP,  _eval_transform)
IMD_dataset    = MultiModalDataset(LOCAL_IMD_AUTH,    LOCAL_IMD_MANIP,    _eval_transform)
RAISE5_dataset = MultiModal5Dataset(LOCAL_RAISE_AUTH, LOCAL_RAISE_MANIP,  _eval_transform)
IMD5_dataset   = MultiModal5Dataset(LOCAL_IMD_AUTH,   LOCAL_IMD_MANIP,    _eval_transform)
_BATCH = 16
RAISE_loader  = DataLoader(RAISE_dataset,  batch_size=_BATCH, shuffle=False)
IMD_loader    = DataLoader(IMD_dataset,    batch_size=_BATCH, shuffle=False)
RAISE5_loader = DataLoader(RAISE5_dataset, batch_size=_BATCH, shuffle=False)
IMD5_loader   = DataLoader(IMD5_dataset,   batch_size=_BATCH, shuffle=False)
print(f"RAISE: {len(RAISE_dataset)} images | IMD2020: {len(IMD_dataset)} images")

# ---- Load all models from Drive ----
print("\nLoading model weights from Drive...")

gating_model = MultiModalFusionNet(freeze_until=1).to(device)
gating_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_gating_fusion.pth", map_location=device))
gating_model.eval()
print("  [OK] MultiModalFusionNet (gating)")

adaptive_model = AdaptiveFusionNet(freeze_until=4).to(device)
adaptive_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_adaptive_fusion.pth", map_location=device))
adaptive_model.eval()
print("  [OK] AdaptiveFusionNet")

vit_model = ViTMultiModalFusionNet(freeze_until=1).to(device)
vit_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_vit_fusion.pth", map_location=device))
vit_model.eval()
print("  [OK] ViTMultiModalFusionNet")

cross_model = CrossModalFusionNet(freeze_until=1).to(device)
cross_model.load_state_dict(torch.load(f"{DRIVE_MODEL_DIR}/best_cross_modal.pth", map_location=device))
cross_model.eval()
print("  [OK] CrossModalFusionNet")

# CASIA2 F1 re-evaluation (optional -- uncomment if g_f1/a_f1/v_f1/x_f1 not in memory)
# full_dataset = MultiModalDataset(LOCAL_CASIA_AUTH, LOCAL_CASIA_MANIP, _eval_transform)
# _, _, test_ds = random_split(full_dataset,
#     [int(0.70*len(full_dataset)), int(0.15*len(full_dataset)),
#      len(full_dataset)-int(0.70*len(full_dataset))-int(0.15*len(full_dataset))],
#     generator=torch.Generator().manual_seed(42))
# test_loader = DataLoader(test_ds, batch_size=_BATCH, shuffle=False)
# full5_dataset = MultiModal5Dataset(LOCAL_CASIA_AUTH, LOCAL_CASIA_MANIP, _eval_transform)
# _, _, test5_ds = random_split(full5_dataset,
#     [int(0.70*len(full5_dataset)), int(0.15*len(full5_dataset)),
#      len(full5_dataset)-int(0.70*len(full5_dataset))-int(0.15*len(full5_dataset))],
#     generator=torch.Generator().manual_seed(42))
# test5_loader = DataLoader(test5_ds, batch_size=_BATCH, shuffle=False)
# _,_,_,g_f1,_,_ = evaluate_model(gating_model,   test_loader,  'Gating - CASIA2')
# _,_,_,a_f1,_,_ = evaluate_model(adaptive_model, test_loader,  'Adaptive - CASIA2')
# _,_,_,v_f1,_,_ = evaluate_model(vit_model,      test_loader,  'ViT Fusion - CASIA2')
# _,_,_,x_f1,_,_ = evaluate_model(cross_model,    test5_loader, 'CrossModal - CASIA2', five_stream=True)

# ---- Option E: Cross-Dataset Generalization Evaluation ----
print("\n" + "="*60 + "\nCross-dataset: RAISE\n" + "="*60)
g_r_acc,_,_,g_r_f1,g_r_auc,_ = evaluate_model(gating_model,   RAISE_loader,  "Gating - RAISE")
a_r_acc,_,_,a_r_f1,a_r_auc,_ = evaluate_model(adaptive_model, RAISE_loader,  "Adaptive - RAISE")
v_r_acc,_,_,v_r_f1,v_r_auc,_ = evaluate_model(vit_model,      RAISE_loader,  "ViT Fusion - RAISE")
x_r_acc,_,_,x_r_f1,x_r_auc,_ = evaluate_model(cross_model,    RAISE5_loader, "CrossModal - RAISE", five_stream=True)

print("\n" + "="*60 + "\nCross-dataset: IMD2020\n" + "="*60)
g_i_acc,_,_,g_i_f1,g_i_auc,_ = evaluate_model(gating_model,   IMD_loader,   "Gating - IMD2020")
a_i_acc,_,_,a_i_f1,a_i_auc,_ = evaluate_model(adaptive_model, IMD_loader,   "Adaptive - IMD2020")
v_i_acc,_,_,v_i_f1,v_i_auc,_ = evaluate_model(vit_model,      IMD_loader,   "ViT Fusion - IMD2020")
x_i_acc,_,_,x_i_f1,x_i_auc,_ = evaluate_model(cross_model,    IMD5_loader,  "CrossModal - IMD2020", five_stream=True)

# ---- Generalisation Gap Analysis ----
# g_f1/a_f1/v_f1/x_f1 must be in memory (from training session or re-eval block above)
gen_df = pd.DataFrame({
    "Model":       ["3-Stream Gating", "3-Stream Adaptive", "ViT Fusion", "CrossModal (A+B)"],
    "CASIA2 F1":   [g_f1,  a_f1,  v_f1,  x_f1],
    "RAISE F1":    [g_r_f1, a_r_f1, v_r_f1, x_r_f1],
    "RAISE Gap":   [g_f1-g_r_f1, a_f1-a_r_f1, v_f1-v_r_f1, x_f1-x_r_f1],
    "IMD2020 F1":  [g_i_f1, a_i_f1, v_i_f1, x_i_f1],
    "IMD2020 Gap": [g_f1-g_i_f1, a_f1-a_i_f1, v_f1-v_i_f1, x_f1-x_i_f1],
})
print("\n===== Generalisation Gap (CASIA2 F1 - Out-of-distribution F1) =====")
print("Smaller gap = better generalisation")
print(gen_df.to_string(index=False, float_format="{:.4f}".format))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
models = gen_df["Model"].tolist()
x      = np.arange(len(models))
width  = 0.35
colors_in  = ["#2ecc71","#e74c3c","#9b59b6","#e67e22"]
colors_out = ["#27ae60","#c0392b","#8e44ad","#d35400"]

for ax, (col_in, col_out, dataset) in zip(axes, [
    ("CASIA2 F1", "RAISE F1",   "RAISE"),
    ("CASIA2 F1", "IMD2020 F1", "IMD2020")
]):
    ax.bar(x - width/2, gen_df[col_in],  width, label="CASIA2 (in-dist)",        color=colors_in,  alpha=0.85)
    ax.bar(x + width/2, gen_df[col_out], width, label=f"{dataset} (out-of-dist)", color=colors_out, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=15, ha="right")
    ax.set_ylabel("F1 Score"); ax.set_ylim(0, 1)
    ax.set_title(f"Generalisation: CASIA2 vs {dataset}", fontsize=13, fontweight="bold")
    ax.legend()
    col_gap = "RAISE Gap" if dataset == "RAISE" else "IMD2020 Gap"
    for i, gap in enumerate(gen_df[col_gap]):
        ax.text(i, max(gen_df[col_in].iloc[i], gen_df[col_out].iloc[i]) + 0.02,
                f"Gap:{gap:.3f}", ha="center", fontsize=9, color="black", fontweight="bold")

plt.suptitle("Cross-Dataset Generalisation Analysis (Option E)", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()
print("\nKey insight: smaller gap indicates manipulation-specific (not dataset-specific) features.")


## Section 12: Ablation Results Table

Complete comparison across all 5 approaches on CASIA2, RAISE, and IMD2020.

In [ ]:
results_df = pd.DataFrame({
    "Model":      ["RF (ELA)", "Single-Stream", "3-Stream Gating", "3-Stream Adaptive", "ViT Fusion", "CrossModal (A+B)"],
    "CASIA2 F1":  [rf_f1,      ss_f1,           g_f1,              a_f1,                v_f1,          x_f1],
    "CASIA2 AUC": [rf_auc,     ss_auc,          g_auc,             a_auc,               v_auc,         x_auc],
    "RAISE F1":   [None,       None,            g_r_f1,            a_r_f1,              v_r_f1,        x_r_f1],
    "IMD2020 F1": [None,       None,            g_i_f1,            a_i_f1,              v_i_f1,        x_i_f1],
})
print(results_df.to_string(index=False, float_format="{:.4f}".format))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = ["#95a5a6","#3498db","#2ecc71","#e74c3c","#9b59b6","#e67e22"]
for ax, col in zip(axes, ["CASIA2 F1","RAISE F1","IMD2020 F1"]):
    vals = results_df[col].fillna(0).tolist()
    bars = ax.bar(results_df["Model"], vals, color=colors)
    ax.set_title(col, fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1); ax.set_ylabel("F1 Score"); ax.tick_params(axis="x", rotation=30)
    for bar, v in zip(bars, vals):
        if v > 0: ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontsize=8)
plt.suptitle("Ablation Study: F1 Across All Models and Datasets", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.show()

## Section 13: Inference Latency Profiling

Measure per-image latency to assess real-time deployability.

In [ ]:
def measure_latency(model, loader, n_batches=50, single_stream=False, five_stream=False, label=""):
    model.eval(); times = []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches: break
            if single_stream:
                imgs, _ = batch; t0 = time.time(); model(imgs.to(device)); n = imgs.size(0)
            elif five_stream:
                rgb,ela,noise,dct,_ = batch
                t0 = time.time(); model(rgb.to(device),ela.to(device),noise.to(device),dct.to(device))
                n = rgb.size(0)
            else:
                rgb,noise,dct,_ = batch
                t0 = time.time(); model(rgb.to(device),noise.to(device),dct.to(device))
                n = rgb.size(0)
            if device.type == "cuda": torch.cuda.synchronize()
            times.append((time.time()-t0)/n)
    ms = np.mean(times) * 1000
    print(f"{label:35s}: {ms:.2f} ms/image")
    return ms

print("\n===== Inference Latency =====")
ss_lat = measure_latency(ss_model,       ss_test_loader, single_stream=True,  label="Single-Stream CNN")
g_lat  = measure_latency(gating_model,   test_loader,                          label="3-Stream Gating")
a_lat  = measure_latency(adaptive_model, test_loader,                          label="3-Stream Adaptive")
v_lat  = measure_latency(vit_model,      test_loader,                          label="ViT Fusion (4-stream)")
x_lat  = measure_latency(cross_model,    test5_loader,   five_stream=True,     label="CrossModal Fusion (A+B)")

lat_df = pd.DataFrame({
    "Model":              ["Single-Stream CNN","3-Stream Gating","3-Stream Adaptive","ViT Fusion","CrossModal (A+B)"],
    "Latency (ms/img)":  [ss_lat, g_lat, a_lat, v_lat, x_lat],
    "Streams":           [1, 3, 3, 4, 5],
})
print("\n", lat_df.to_string(index=False, float_format="{:.2f}".format))

# Bar chart
plt.figure(figsize=(9, 5))
colors = ["#3498db","#2ecc71","#e74c3c","#9b59b6","#e67e22"]
bars = plt.bar(lat_df["Model"], lat_df["Latency (ms/img)"], color=colors)
for bar, v in zip(bars, lat_df["Latency (ms/img)"]):
    plt.text(bar.get_x()+bar.get_width()/2, v+0.1, f"{v:.1f}ms", ha="center", fontsize=9)
plt.axhline(y=33, color="red", linestyle="--", alpha=0.5, label="30fps threshold (33ms)")
plt.ylabel("ms / image"); plt.title("Inference Latency per Model", fontsize=13, fontweight="bold")
plt.xticks(rotation=20, ha="right"); plt.legend(); plt.tight_layout(); plt.show()

## Section 14: 3-Class Fine-Tuning — AI-Generated Image Detection

The original CrossModalFusionNet was trained on CASIA2 (traditional manipulation: copy-move, splicing).
AI-generated images have no JPEG re-compression artifacts so all forensic streams return "clean"
readings, causing the model to misclassify them as Authentic.

**Strategy:** Fine-tune CrossModalFusionNet with a 3-class output head on a mixed dataset:
- Class 0 — Authentic (CASIA2 real images)
- Class 1 — Traditionally Manipulated (CASIA2 tampered images)
- Class 2 — AI-Generated (CIFAKE FAKE split)

**Partial fine-tune** (user choice): only `modality_transformer` + `classifier` are updated.
All 4 EfficientNet-B0 backbones and the ViT encoder are frozen — this preserves the forensic
feature representations learned during original training and prevents catastrophic forgetting.

### Setup: Download CIFAKE from Kaggle

Run this once per Colab session before the training cell:

```python
# 1. Upload your kaggle.json API key to Colab, then:
!pip install kaggle -q
import os, shutil
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# 2. Download and unzip CIFAKE
!kaggle datasets download -d jordandavis/cifake -p /content/cifake --unzip -q

# 3. Verify
import os
fake_dir = "/content/cifake/test/FAKE"
print(f"CIFAKE FAKE images: {len(os.listdir(fake_dir))}")
```

Alternatively, upload your own folder of AI-generated images and set `AI_GENERATED_DIR` below.

In [ ]:
# ============================================================
# Section 14A: Pre-cache modality tensors + ThreeClassDatasetCached
# ============================================================
# WHY: Computing ELA (disk write + JPEG re-read), noise residual, and DCT
#      inside __getitem__ costs ~3-4 s/image on CPU (I/O + recompute every
#      epoch). We pay that cost ONCE here, saving each image's 4 tensors to
#      a .pt file. __getitem__ then becomes a cheap torch.load() call,
#      dropping per-epoch data time from ~43 min → ~2-3 min.
#
# Prerequisites:
#   - CASIA2 copied to local VM (Section 3A)
#   - CIFAKE train/FAKE uploaded to Drive; set AI_GENERATED_DIR below
# ============================================================

import hashlib

# ---- Inference transform (no augmentation — modalities stored normalised) ----
cache_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ---- extract_streams helper (mirrors app/preprocess.py exactly) ----
def extract_streams(img):
    """Return (rgb, ela, noise, dct) — each (1,3,224,224) float32 tensor."""
    rgb   = cache_transform(img).unsqueeze(0)
    ela   = cache_transform(get_ela_modality(img)).unsqueeze(0)
    noise = cache_transform(get_noise_residual(img)).unsqueeze(0)
    dct   = cache_transform(get_dct_image(img)).unsqueeze(0)
    return rgb, ela, noise, dct

# ---- Path to AI-generated images ----
AI_GENERATED_DIR = "/content/cifake/train/FAKE"   # <-- adjust if needed

# Cache lives on Colab's local SSD (/content) — fastest I/O.
# Re-run this cell after a session restart to rebuild the cache.
# To persist across sessions, change CACHE_DIR to a Drive path
# (e.g. f"{DRIVE_MODEL_DIR}/modality_cache") but expect ~3× slower reads.
CACHE_DIR = "/content/modality_cache"

# Verify all three directories exist
for _dir, _name in [
    (LOCAL_CASIA_AUTH,  "CASIA2 Authentic"),
    (LOCAL_CASIA_MANIP, "CASIA2 Tampered"),
    (AI_GENERATED_DIR,  "AI-Generated"),
]:
    n = len([f for f in os.listdir(_dir) if f.lower().endswith(VALID_EXT)])
    print(f"  {_name}: {n} images  →  {_dir}")


# ---- Cache builder ----
def build_cache(src_dir: str, label: int, cache_dir: str, max_images: int = None):
    """
    Pre-compute (rgb, ela, noise, dct) for every image in src_dir and save
    each as {cache_dir}/{label}/{stem_hash}.pt.
    Skips files that already exist — safe to resume after interruption.
    Returns list of (pt_path, label) tuples.
    """
    out_dir = os.path.join(cache_dir, str(label))
    os.makedirs(out_dir, exist_ok=True)

    paths = sorted([
        os.path.join(src_dir, f)
        for f in os.listdir(src_dir)
        if f.lower().endswith(VALID_EXT)
    ])
    if max_images:
        random.shuffle(paths)
        paths = paths[:max_images]

    label_names = {0: "Authentic", 1: "Manipulated", 2: "AI-Generated"}
    samples, skipped = [], 0

    for p in tqdm(paths, desc=f"Caching {label_names[label]}"):
        stem   = os.path.splitext(os.path.basename(p))[0]
        uid    = stem + "_" + hashlib.md5(p.encode()).hexdigest()[:6]
        out_pt = os.path.join(out_dir, uid + ".pt")

        if not os.path.exists(out_pt):
            try:
                img = Image.open(p).convert("RGB")
                rgb, ela, noise, dct = extract_streams(img)
                torch.save({
                    "rgb":   rgb.squeeze(0),    # (3, 224, 224) float32
                    "ela":   ela.squeeze(0),
                    "noise": noise.squeeze(0),
                    "dct":   dct.squeeze(0),
                }, out_pt)
            except Exception as e:
                skipped += 1
                continue

        samples.append((out_pt, label))

    print(f"  → {len(samples)} cached  |  {skipped} skipped/errored")
    return samples


# ---- Balanced class cap ----
_n_auth  = len([f for f in os.listdir(LOCAL_CASIA_AUTH)  if f.lower().endswith(VALID_EXT)])
_n_manip = len([f for f in os.listdir(LOCAL_CASIA_MANIP) if f.lower().endswith(VALID_EXT)])
_n_ai    = len([f for f in os.listdir(AI_GENERATED_DIR)  if f.lower().endswith(VALID_EXT)])
_cap = min(_n_auth, _n_manip, _n_ai)
print(f"\nClass sizes: Auth={_n_auth} | Manip={_n_manip} | AI={_n_ai}")
print(f"Capping each class at {_cap} images.\n")

# ---- Build caches (runs once; skips existing files on re-run) ----
samples_auth  = build_cache(LOCAL_CASIA_AUTH,  0, CACHE_DIR, max_images=_cap)
samples_manip = build_cache(LOCAL_CASIA_MANIP, 1, CACHE_DIR, max_images=_cap)
samples_ai    = build_cache(AI_GENERATED_DIR,  2, CACHE_DIR, max_images=_cap)

all_samples = samples_auth + samples_manip + samples_ai
random.shuffle(all_samples)


# ---- Dataset — __getitem__ is now just torch.load() ----
class ThreeClassDatasetCached(Dataset):
    def __init__(self, samples):
        self.samples = samples   # list of (pt_path: str, label: int)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pt_path, label = self.samples[idx]
        data = torch.load(pt_path, weights_only=True)
        return data["rgb"], data["ela"], data["noise"], data["dct"], label


# ---- Train / val split (80/20) ----
_split        = int(0.80 * len(all_samples))
train_samples = all_samples[:_split]
val_samples   = all_samples[_split:]

# Doubled batch size (safe now that preprocessing is off the critical path).
# num_workers=2 for async prefetch; set to 0 if you hit "BrokenPipeError".
BATCH3 = 32
train3_dataset = ThreeClassDatasetCached(train_samples)
val3_dataset   = ThreeClassDatasetCached(val_samples)

train3_loader = DataLoader(train3_dataset, batch_size=BATCH3, shuffle=True,
                           num_workers=2, pin_memory=True)
val3_loader   = DataLoader(val3_dataset,   batch_size=BATCH3, shuffle=False,
                           num_workers=2, pin_memory=True)

print(f"\n3-class dataset — Train: {len(train3_dataset)} | Val: {len(val3_dataset)}")
print("Label distribution (train):")
from collections import Counter
_dist = Counter(lbl for _, lbl in train_samples)
for _c, _name in [(0, "Authentic"), (1, "Manipulated"), (2, "AI-Generated")]:
    print(f"  Class {_c} ({_name}): {_dist[_c]}")

In [ ]:
# ============================================================
# Section 14B: 3-Class CrossModalFusionNet + Fine-Tuning
# ============================================================
# Loads the 2-class checkpoint, replaces the final head with a
# fresh 3-class linear layer, then fine-tunes only:
#   - modality_transformer (cross-modal fusion)
#   - classifier (final head)
# All EfficientNet-B0 backbones and ViT are frozen.
# ============================================================

# ---- Step 1: Load 2-class checkpoint into base architecture ----
cross3_model = CrossModalFusionNet(freeze_until=1, d_model=1280).to(device)
_ckpt_path = f"{DRIVE_MODEL_DIR}/best_cross_modal.pth"
_state = torch.load(_ckpt_path, map_location=device)
if isinstance(_state, dict) and "model_state_dict" in _state:
    _state = _state["model_state_dict"]
cross3_model.load_state_dict(_state)
print(f"Loaded 2-class weights from {_ckpt_path}")

# ---- Step 2: Replace final Linear(256 → 2) with Linear(256 → 3) ----
# The new head is randomly initialised; all other weights are preserved.
cross3_model.classifier[-1] = nn.Linear(256, 3).to(device)
print("Replaced classifier head: 256 → 3 classes")

# ---- Step 3: Freeze everything EXCEPT modality_transformer + classifier ----
for name, param in cross3_model.named_parameters():
    trainable = name.startswith("modality_transformer") or name.startswith("classifier")
    param.requires_grad = trainable

trainable_params = sum(p.numel() for p in cross3_model.parameters() if p.requires_grad)
total_params      = sum(p.numel() for p in cross3_model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} parameters "
      f"({100*trainable_params/total_params:.1f}%)")

# ---- Step 4: Optimiser — only trainable params ----
ft_opt = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, cross3_model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
ft_sch  = optim.lr_scheduler.ReduceLROnPlateau(ft_opt, mode="max", factor=0.5, patience=2)
ft_crit = nn.CrossEntropyLoss(label_smoothing=0.05)

# ---- Step 5: Fine-tuning loop ----
FINETUNE_EPOCHS = 10
best_ft_acc, ft_pat = 0.0, 0

for epoch in range(FINETUNE_EPOCHS):
    cross3_model.train()
    ep_loss = 0
    for rgb, ela, noise, dct, lbls in tqdm(train3_loader, desc=f"FT Epoch {epoch+1}/{FINETUNE_EPOCHS}"):
        ft_opt.zero_grad()
        logits = cross3_model(rgb.to(device), ela.to(device), noise.to(device), dct.to(device))
        loss   = ft_crit(logits, lbls.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cross3_model.parameters(), 1.0)
        ft_opt.step()
        ep_loss += loss.item()

    # Validation
    cross3_model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for rgb, ela, noise, dct, lbls in val3_loader:
            preds = torch.argmax(
                cross3_model(rgb.to(device), ela.to(device), noise.to(device), dct.to(device)), dim=1
            )
            vp.extend(preds.cpu().numpy())
            vl.extend(lbls.numpy())

    from sklearn.metrics import accuracy_score as _acc
    val_acc = _acc(vl, vp)
    ft_sch.step(val_acc)
    print(f"Epoch {epoch+1} | Loss: {ep_loss/len(train3_loader):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_ft_acc:
        best_ft_acc = val_acc
        torch.save(cross3_model.state_dict(), f"{DRIVE_MODEL_DIR}/best_cross_modal_3class.pth")
        ft_pat = 0
        print(f"  ✓ Saved best model (val acc {val_acc:.4f})")
    else:
        ft_pat += 1
    if ft_pat >= 3:
        print("Early stop.")
        break

print(f"\nFine-tuning complete. Best val accuracy: {best_ft_acc:.4f}")
print(f"Checkpoint saved to: {DRIVE_MODEL_DIR}/best_cross_modal_3class.pth")

In [ ]:
# ============================================================
# Section 14C: Evaluate 3-Class Model
# ============================================================

cross3_model.load_state_dict(
    torch.load(f"{DRIVE_MODEL_DIR}/best_cross_modal_3class.pth", map_location=device)
)
cross3_model.eval()

CLASS_NAMES_3 = ["Authentic", "Manipulated", "AI-Generated"]

all_preds, all_labels = [], []
with torch.no_grad():
    for rgb, ela, noise, dct, lbls in tqdm(val3_loader, desc="Evaluating 3-class model"):
        logits = cross3_model(rgb.to(device), ela.to(device), noise.to(device), dct.to(device))
        preds  = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.numpy())

print("\n===== 3-Class CrossModalFusionNet =====")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES_3, digits=4))

# Confusion matrix
cm3 = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm3, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES_3, yticklabels=CLASS_NAMES_3)
plt.title("3-Class CrossModalFusionNet — Confusion Matrix")
plt.ylabel("True label"); plt.xlabel("Predicted label")
plt.tight_layout(); plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(CLASS_NAMES_3):
    mask = np.array(all_labels) == i
    acc_i = (np.array(all_preds)[mask] == i).mean()
    print(f"  {name}: {acc_i:.4f}")